In [46]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, confusion_matrix, accuracy_score

# Generalized Linear Models

So far, we have used linear regression for cases where the task at hand is to predict a continuous value. However, the predictive power of the linear model can be extended to tasks such as classification or the prediction of discrete values. Here, we will look at how logistic regression models work and how they are used for classification tasks.

## Warmup

The common linear model is described in this way:

$$z = \beta_0 + \beta_1x_1 + \beta_2x_2 + ... + \beta_nx_n$$

It allows the target value $z$ to be in the domain of real numbers, i.e., to have values in $[-\infty, \infty]$. However, when we are dealing with binary classification tasks, we need a restricted range, for example $[0,1]$, where $0$ and $1$ represent the possible classes. For this reason, we take the output of the linear function and pass it through the sigmoid function:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

This function maps the output to the interval $[0,1]$. For large positive values of $z$, the term $e^{-z}$ will be close to $0$ and $\sigma(z)$ will be close to $1$. The same happens in the opposite direction: when $z$ takes large negative values, $e^{-z}$ becomes a huge number and the result approaches $0$. And therefore we can set a threshold, for instance $0.5$, so the observations with a predicted value above this threshold will pertain to class $1$, and below to class $0$.

### a)

Plot the sigmoid function for values of $z$ between $-10$ and $10$. Which value does it get when $z = -10$? Can you see how this function restricts the range to be between $0$ and $1$? Which is the value when $z=0$?

## Exercise 1
### Logistic regression

We will now introduce *classification* in a regression setting, reusing the same birth-weight data set as in the previous week.

### a)

- Fit a logistic regression model with `gender` as the response, using all the other covariates as predictors. You can use [`sm.Logit`](https://www.statsmodels.org/stable/generated/statsmodels.discrete.discrete_model.Logit.html) from `statsmodels`.

- Print the model summary. Looking at the $p-values$ which variables do you consider to be relevant?

In [48]:
data = pd.read_csv('https://www.uio.no/studier/emner/matnat/math/STK1110/data/fodsler.txt', sep=r'\s+', engine='python')
data.rename(columns={'Kjonn': 'gender', 'SvDager': 'pregnancy_days', 'MorsAld': 'mother_age', 'Antfod': 'n_children', 'Fvekt': 'birth_weight'}, inplace=True)
data

,gender,pregnancy_days,mother_age,n_children,birth_weight
0,1,288,24,1,4070
1,1,286,27,0,4220
2,0,276,36,2,3690
3,1,265,31,0,3480
4,1,278,24,1,3970
...,...,...,...,...,...
995,0,273,25,0,3470
996,0,299,30,2,4010
997,1,267,25,3,4020
998,0,275,27,0,2910


### b)
Comment on the fit of your model. If you do not consider all covariates relevant, refit a smaller model with only the ones you want to keep, and compare the $AIC$.

### c)
The odds ratio and log-odds are two terms we make use of related to logistic regression.

- In your own words, define what they mean, and write down how they relate to a coefficient $\beta_j$ in a logistic regression model.

- For your final logistic regression model, program a way to report the estimates of these.

## Exercise 2

In this exercise we aim to predict the presence of a gene associated with the development of haemochromatosis (excess iron absorption during digestion). The variables in the Hemocromatosis.csv file are as follows:
- Gen: 0 – absence, 1 – presence

- Ferr: Serum ferritin level (µg/L)

- TSI: Transferrin saturation index (%)

- North: 0 – southern, 1 – northern

In [52]:
gen = pd.read_csv("Hemocromatosis.csv")
gen.tail()

,Gen,Ferr,TSI,North
75,1,860,58,1
76,1,1211,48,0
77,1,1014,67,1
78,1,888,58,0
79,1,691,86,0


### a)

Create a logistic regression model to explain the presence of the gene associated with haemochromatosis based on the other three variables in the database.

### b)

- Calculate the odds ratio (OR, i.e. the odds multiplied by) if a person is from the north rather than from the south.
- Calculate the OR associated with a 100 µg/L difference in ferritin between two individuals.
- In each case, indicate which of the two hypothetical individuals is more likely to carry the gene in question.

When performing classification tasks, we use the confusion matrix to evaluate our model's predictive performance. In this matrix, the main diagonal contains the correct predictions, while the remaining cells represent the errors (misclassifications). By using the values from this matrix, we can calculate metrics such as specificity and sensitivity. Although these two metrics are related, they measure different aspects of the model and are calculated as follows:

Sensitivity (Recall)It measures the proportion of actual positives that were correctly identified:
$$\text{Sensitivity} = \frac{TP}{TP + FN}$$

SpecificityIt measures the proportion of actual negatives that were correctly identified:
$$\text{Specificity} = \frac{TN}{TN + FP}$$

### c)

Using the threshold of $0.5$, compute the confusion matrix, specificity, and sensitivity of the model using the provided data.

<span style="color:red">You can use the function `sklearn.metrics.confusion_matrix()`</span>

### d)
Explain why we might prioritise Sensitivity and when Specificity.

The ROC curve allows us to visualise both metrics simultaneously, and shows how sensitivity changes as we sacrifice specificity. And the area under this curve can tell us how good the model is overall.

### e)
Computes and displays the ROC curve, calculates the area under the curve, and determines the cut-off point with the threshold of $p=0.5$ and the optimal point on the graph, which is the point closest to the top-left corner.

### f)
What do you consider about the model? How far are the two points in the curve? Based on the area under the curve, is it a good model? Analyse your result.

## Exercise 3
### Cross Validation

Now that we already learnt how to create logistic models, in this section we will evaluate and compare them, for finally select the set of variables with best prediction capacity.

In this exercise we will use a dataset to predict if a person sufferd from a stroke based on many other variables. The dataset is loaded and irrelevant columns are dropped. As you can see, there are both continuous and categorical values.

In [57]:
stroke = pd.read_csv("healthcare-dataset-stroke-data.csv")
stroke.drop(columns=['bmi', "id", "hypertension", "heart_disease"], inplace=True)
stroke

,gender,age,ever_married,work_type,Residence_type,avg_glucose_level,smoking_status,stroke
0,Male,67.0,Yes,Private,Urban,228.69,formerly smoked,1
1,Female,61.0,Yes,Self-employed,Rural,202.21,never smoked,1
2,Male,80.0,Yes,Private,Rural,105.92,never smoked,1
3,Female,49.0,Yes,Private,Urban,171.23,smokes,1
4,Female,79.0,Yes,Self-employed,Rural,174.12,never smoked,1
...,...,...,...,...,...,...,...,...
5105,Female,80.0,Yes,Private,Urban,83.75,never smoked,0
5106,Female,81.0,Yes,Self-employed,Urban,125.20,never smoked,0
5107,Female,35.0,Yes,Self-employed,Rural,82.99,never smoked,0
5108,Male,51.0,Yes,Private,Rural,166.29,formerly smoked,0


### a)
We will start by a little bit of data visualization. Start by ploting the counts of each value of column `stroke`. If we had a model that always predicts category $0$, what would be the accuracy of that model? Why might this happen?

### b)
Create a logistic regression model with all the variables of the dataset. You should use `pd.get_dummies()` and convert the variables to float with `.astype(float)` to correctly preprocess the categorical variables.

Once you create the dummy variables, add the constant and create the model you will see that the model is composed by many variables, many of them with very low p-values associated. What are these variables?

### c)
Create now a model as before but only with the continuous variables, i.e. `age` and `avg_glucose_level`.

The cross-validation method is used to validate different models, select variables and assess predictive ability. It involves first taking the dataset and dividing it into several folds – let’s say 10, for example – and in each iteration using 9 of the folds to estimate the parameters, whilst using the last fold to validate the model’s predictive ability on a dataset not seen during training.

### d)
Create a function that takes the next parameters as input:
- X: The covariates matrix
- y: The target variable
- n_splits: The number of folds
- threshold: The value that we will use to asses if the prediction is 1 or 0

And the function must perform cross-validation; in each iteration, it must calculate the accuracy, sensitivity and specificity of the validation fold, and it must return the result of each iteration as well as the average of all the results.

You can use the function `sklearn.model_selection.StratifiedKFold()` to divide the data.

### e)
Now, apply this function to:
- The set of only continuous variables
- The continuous variables with `Residence_type` and `smoking_status`

What is the sensitivity and specificity when the threshold is $0.5$?

Now do the same but with other values of the threshold. Do you see any change? Why is that happening?

Do these categorical variables we are adding improve or worsen the outcome?